In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import torch
import torch.nn as nn
from torch.optim import AdamW
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
import warnings
warnings.filterwarnings('ignore')


# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")
print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
r_path = os.path.join(path, 'Q1_data.csv')
rest = pd.read_csv(r_path)

In [ ]:
# Task 2: Write your code here:
print(f"Shape: {rest.shape}")
rest.head()

In [ ]:
# Task 3: Write your code here:
rest.info()

In [ ]:
# Task 4: Write your code here:
rest.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(8, 6))

def check_target_distribution(rest, Delivery_Time):
  rest[Delivery_Time].hist(bins=10, edgecolor='black')

  plt.title(f"Target Distribution ({Delivery_Time})")
  plt.xlabel(Delivery_Time)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(rest, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
#rest.columns.drop("Order_ID")
#print(f"Shape after cleaning: {rest.shape}")
stat_cols = ['Order_ID', 'Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs', 'Delivery_Time']

rest_clean = rest.dropna(subset=stat_cols).copy()
print(f"Shape after cleaning: {rest_clean.shape}")

print("Missing values:")
print(rest.isnull().sum())

def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(rest)

rest.drop_duplicates(inplace=True)

In [ ]:
# Task 2: Write your code here:
rest.fillna('none')

In [ ]:
# Task 3: Write your code here:
# Encode type columns
stat_cols = ['Order_ID', 'Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs', 'Delivery_Time']

In [ ]:
# Task 4: Write your code here:
le = LabelEncoder()
rest['Delivery_Time'] = le.fit_transform(rest['Delivery_Time'])
rest['Vehicle_Type'] = le.fit_transform(rest['Vehicle_Type'])

In [ ]:
# Task 5: Write your code here:
categorical_cols = rest.select_dtypes(include=["object"]).columns

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm
print("Categorical Columns:", list(categorical_cols))

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  rest[col] = le.fit_transform(rest[col])
  label_encoders[col] = le

rest

In [ ]:
# Task 6: Write your code here:
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(rest, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
X = rest.drop("Delivery_Time", axis=1).astype(float)
y = rest['Delivery_Time'].astype(float)

In [ ]:
# sigmoid in NumPy
def sigmoid(z):
  return 1 / (1 + np.exp(-z))
# BCE in NumPy
def binary_cross_entropy(y, y_hat):
  epsilon = 1e-15  # Very small number to prevent log(0)
  y_hat = np.clip(y_hat, epsilon, 1 - epsilon) # np.clip(value, min, max)

  loss = -1/len(y) * np.sum(y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat))
  return loss
def gradient_descent(X, y, learning_rate, n_iters=500):
  m, n = X.shape # m rows, n columns (dimensions)
  theta = np.zeros(n) # initialize a zeros weight vector with n dimensions
  losses = []

  for _ in tqdm(range(n_iters), desc="Training Logistic Regression"):
    z = np.dot(X, theta)
    y_hat = sigmoid(z)
    gradient = np.dot(X.T, (y_hat - y)) / m
    theta -= learning_rate * gradient

    loss = binary_cross_entropy(y, y_hat)
    losses.append(loss)

  return theta, losses


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
n_splits = 5 # K=5 Folds

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

In [ ]:
# Storage for logistic regression results for each fold
lr_losses = []
lr_accuracy = []
lr_precision = []
lr_recall = []
lr_f1 = []

In [ ]:
for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
    print(f"\nFold {fold_idx + 1}/{n_splits}")

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Train
    theta, losses = gradient_descent(X_train, y_train, learning_rate=0.5, n_iters=500)

    # Validate
    y_pred_proba = sigmoid(np.dot(X_test, theta))
    y_pred = (y_pred_proba >= 0.5).astype(int)

    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    # Store results
    lr_losses.append(losses)
    lr_accuracy.append(accuracy)
    lr_precision.append(precision)
    lr_recall.append(recall)
    lr_f1.append(f1)

In [ ]:
# Calculate average loss across folds
avg_loss = np.mean(lr_losses, axis=0)

plt.figure(figsize=(10, 6))
plt.plot(avg_loss, label='Logistic Regression Loss', color='purple')
plt.title('Average Logistic Regression Loss Curve (Across 5 Folds)')
plt.xlabel('Iteration')
plt.ylabel('Binary Cross-Entropy Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Task 1: Write your code here:
# Gather importances from the models (from the last fold)
importances = {}

importances['Random Forest'] = sklearn_models['Random Forest'].feature_importances_
importances['XGBoost'] = sklearn_models['XGBoost'].feature_importances_
importances['CatBoost'] = sklearn_models['CatBoost'].feature_importances_

# Create a 1x3 plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, imp) in enumerate(importances.items()):
  # Sort features by importance for a cleaner plot
  sorted_idx = np.argsort(imp)

  ax = axes[i]
  ax.barh(features[sorted_idx], imp[sorted_idx])
  ax.set_title(f"{model_name} Feature Importance")
  ax.set_xlabel("Importance Score")

plt.tight_layout()
plt.show()

In [ ]:
# Task Bonus: Write your code here: